In [2]:
import numpy as np
import pandas as pd

# Load datasets
master_df = pd.read_csv('../data/constituency_master.csv')
df_2021 = pd.read_csv('../data/tn_2021_results.csv')
df_2026 = pd.read_csv('../data/tn_2026_results.csv')

print(f"Master ACs: {master_df.shape[0]}")
print(f"2021 Rows: {df_2021.shape[0]}, Columns: {df_2021.shape[1]}")
print(f"2026 Rows: {df_2026.shape[0]}, Columns: {df_2026.shape[1]}")

Master ACs: 234
2021 Rows: 4232, Columns: 8
2026 Rows: 4257, Columns: 8


In [3]:
# Check first 3 rows of each table
display(master_df.head(3))
display(df_2021.head(3))
display(df_2026.head(3))

# Check for missing values across all files
print("\n--- Missing values in Master ---")
print(master_df.isnull().sum())

print("\n--- Missing values in 2021 ---")
print(df_2021.isnull().sum())

print("\n--- Missing values in 2026 ---")
print(df_2026.isnull().sum())

,ac_number,constituency,district,region,reserved
0,1,Gummidipundi,Tiruvallur,Chennai Metro,GEN
1,2,Ponneri,Tiruvallur,Chennai Metro,SC
2,3,Tiruttani,Tiruvallur,Chennai Metro,GEN


,constituency,ac_number,candidate,party,votes,turnout,reserved,region
0,Gummidipundi,1,GOVINDARAJAN T.J,DMK,126452,78.08,GEN,Chennai Metro
1,Gummidipundi,1,PRAKASH M,PMK,75514,78.08,GEN,Chennai Metro
2,Gummidipundi,1,USHA,NTK,11701,78.08,GEN,Chennai Metro


,constituency,ac_number,candidate,party,votes,turnout,reserved,region
0,Gummidipoondi,1,S.VIJAYAKUMAR,TVK,94320,91.48,GEN,Chennai Metro
1,Gummidipoondi,1,SUDHAKAR.V,AIADMK,66375,91.48,GEN,Chennai Metro
2,Gummidipoondi,1,T.J.GOVINDARAJAN,DMK,62492,91.48,GEN,Chennai Metro



--- Missing values in Master ---
ac_number       0
constituency    0
district        0
region          0
reserved        0
dtype: int64

--- Missing values in 2021 ---
constituency    0
ac_number       0
candidate       0
party           0
votes           0
turnout         0
reserved        0
region          0
dtype: int64

--- Missing values in 2026 ---
constituency    0
ac_number       0
candidate       0
party           0
votes           0
turnout         0
reserved        0
region          0
dtype: int64


In [4]:
# Function to get the winner for each constituency
def get_constituency_winners(df, year):
    # Sort descending by votes within each ac_number
    df_sorted = df.sort_values(['ac_number', 'votes'], ascending=[True, False])
    
    # Drop duplicates to keep only the highest vote row (winner)
    winners = df_sorted.drop_duplicates(subset=['ac_number'], keep='first').copy()
    
    # Add year column for tracking
    winners['election_year'] = year
    return winners

winners_2021 = get_constituency_winners(df_2021, 2021)
winners_2026 = get_constituency_winners(df_2026, 2026)

print(f"2021 Winners: {len(winners_2021)} ACs")
print(f"2026 Winners: {len(winners_2026)} ACs")

2021 Winners: 234 ACs
2026 Winners: 234 ACs


In [5]:
# Rename columns to avoid confusion after merging
w2021 = winners_2021[['ac_number', 'candidate', 'party', 'votes', 'turnout']].rename(
    columns={'candidate': 'candidate_2021', 'party': 'party_2021', 'votes': 'votes_2021', 'turnout': 'turnout_2021'}
)

w2026 = winners_2026[['ac_number', 'candidate', 'party', 'votes', 'turnout']].rename(
    columns={'candidate': 'candidate_2026', 'party': 'party_2026', 'votes': 'votes_2026', 'turnout': 'turnout_2026'}
)

# Merge Master with 2021
analysis_df = pd.merge(master_df, w2021, on='ac_number', how='left')

# Merge the result with 2026
analysis_df = pd.merge(analysis_df, w2026, on='ac_number', how='left')

# Let's verify the columns and check the first couple of rows
print("Columns in our analysis table:\n", list(analysis_df.columns))
display(analysis_df.head(3))

Columns in our analysis table:
 ['ac_number', 'constituency', 'district', 'region', 'reserved', 'candidate_2021', 'party_2021', 'votes_2021', 'turnout_2021', 'candidate_2026', 'party_2026', 'votes_2026', 'turnout_2026']


,ac_number,constituency,district,region,reserved,candidate_2021,party_2021,votes_2021,turnout_2021,candidate_2026,party_2026,votes_2026,turnout_2026
0,1,Gummidipundi,Tiruvallur,Chennai Metro,GEN,GOVINDARAJAN T.J,DMK,126452,78.08,S.VIJAYAKUMAR,TVK,94320,91.48
1,2,Ponneri,Tiruvallur,Chennai Metro,SC,DURAI. CHANDRASEKAR,INC,94528,78.20,DR.RAVI.M.S,TVK,110439,90.21
2,3,Tiruttani,Tiruvallur,Chennai Metro,GEN,S.Chandran,DMK,120314,78.76,G.HARI,AIADMK,89169,90.97


In [6]:
# Create a boolean column that is True if the seat flipped
analysis_df['seat_flipped'] = analysis_df['party_2021'] != analysis_df['party_2026']

# Count total flips
total_flips = analysis_df['seat_flipped'].sum()
print(f"Total Constituencies Flipped: {total_flips} out of 234")

# Group by the 2021 Party and 2026 Party to see the flow of seats
flip_summary = analysis_df.groupby(['party_2021', 'party_2026']).size().reset_index(name='seat_count')

# Sort to see the biggest shifts at the top
flip_summary = flip_summary.sort_values(by='seat_count', ascending=False)
display(flip_summary.head(10))

Total Constituencies Flipped: 163 out of 234


,party_2021,party_2026,seat_count
17,DMK,TVK,65
14,DMK,DMK,40
4,AIADMK,TVK,26
0,AIADMK,AIADMK,22
11,DMK,AIADMK,22
1,AIADMK,DMK,15
22,INC,TVK,11
21,INC,INC,4
16,DMK,PMK,3
23,PMK,AIADMK,2


In [7]:
# Function to calculate total votes, winner, runner-up, and margins
def calculate_margins(df):
    # 1. Get total valid votes per constituency
    total_votes = df.groupby('ac_number')['votes'].sum().reset_index(name='total_valid_votes')
    
    # 2. Sort candidates by votes descending within each constituency
    df_sorted = df.sort_values(by=['ac_number', 'votes'], ascending=[True, False])
    
    # 3. Assign a rank to each candidate (1 = Winner, 2 = Runner-up)
    df_sorted['rank'] = df_sorted.groupby('ac_number')['votes'].cumcount() + 1
    
    winners = df_sorted[df_sorted['rank'] == 1].copy()
    runners_up = df_sorted[df_sorted['rank'] == 2].copy()
    
    # 4. Merge winner and runner-up data
    margins = pd.merge(winners[['ac_number', 'party', 'votes']], 
                       runners_up[['ac_number', 'party', 'votes']], 
                       on='ac_number', suffixes=('_win', '_ru'))
    
    # 5. Merge with the total valid votes
    margins = pd.merge(margins, total_votes, on='ac_number')
    
    # 6. Calculate Margin and Vote Share Percentages
    margins['margin_votes'] = margins['votes_win'] - margins['votes_ru']
    margins['margin_pct'] = (margins['margin_votes'] / margins['total_valid_votes']) * 100
    margins['win_vote_share_pct'] = (margins['votes_win'] / margins['total_valid_votes']) * 100
    
    return margins

margins_2021 = calculate_margins(df_2021)
margins_2026 = calculate_margins(df_2026)

# Answer Research Question 6
print("2021 - Average Margin %:", round(margins_2021['margin_pct'].mean(), 2))
print("2026 - Average Margin %:", round(margins_2026['margin_pct'].mean(), 2))
print("---")
print("2021 - Seats won with > 50% vote share:", (margins_2021['win_vote_share_pct'] > 50).sum())
print("2026 - Seats won with > 50% vote share:", (margins_2026['win_vote_share_pct'] > 50).sum())
print("2021 - Seats won with < 35% vote share:", (margins_2021['win_vote_share_pct'] < 35).sum())
print("2026 - Seats won with < 35% vote share:", (margins_2026['win_vote_share_pct'] < 35).sum())

2021 - Average Margin %: 11.7
2026 - Average Margin %: 7.67
---
2021 - Seats won with > 50% vote share: 70
2026 - Seats won with > 50% vote share: 13
2021 - Seats won with < 35% vote share: 2
2026 - Seats won with < 35% vote share: 64


In [8]:
# --- 1. Regional and Reserved Seat Flow (Questions 1 & 4) ---
# We already have analysis_df with 2021 and 2026 winners joined to the master table.
# Let's save this as our main "Seat Flow" table.
analysis_df.to_csv('../data/powerbi_seat_flips.csv', index=False)

# --- 2. Turnout Surge (Question 5) ---
# Extracting the turnout per constituency for both years
t2021 = df_2021.groupby('ac_number')['turnout'].first().reset_index(name='turnout_2021')
t2026 = df_2026.groupby('ac_number')['turnout'].first().reset_index(name='turnout_2026')

turnout_df = pd.merge(master_df, t2021, on='ac_number')
turnout_df = pd.merge(turnout_df, t2026, on='ac_number')
turnout_df['turnout_diff'] = turnout_df['turnout_2026'] - turnout_df['turnout_2021']

# Let's look at the top 20 increases to answer the prompt
top20_turnout = turnout_df.sort_values('turnout_diff', ascending=False).head(20)
print("--- Top 20 Turnout Increases by Region ---")
print(top20_turnout['region'].value_counts())

turnout_df.to_csv('../data/powerbi_turnout.csv', index=False)


# --- 3. Vote Share (Question 3) ---
# Calculate total state-wide valid votes for each year
total_votes_21 = df_2021['votes'].sum()
total_votes_26 = df_2026['votes'].sum()

# Aggregate votes by party for 2021
party_votes_21 = df_2021.groupby('party')['votes'].sum().reset_index()
party_votes_21['vote_share_2021'] = (party_votes_21['votes'] / total_votes_21) * 100

# Aggregate votes by party for 2026
party_votes_26 = df_2026.groupby('party')['votes'].sum().reset_index()
party_votes_26['vote_share_2026'] = (party_votes_26['votes'] / total_votes_26) * 100

# Rename to avoid conflicts when joining
party_votes_21.rename(columns={'votes': 'total_votes_21'}, inplace=True)
party_votes_26.rename(columns={'votes': 'total_votes_26'}, inplace=True)

# Merge to create a single vote share comparison table
vote_share_df = pd.merge(party_votes_21, party_votes_26, on='party', how='outer').fillna(0)
vote_share_df = vote_share_df.sort_values('total_votes_26', ascending=False)
vote_share_df.to_csv('../data/powerbi_vote_shares.csv', index=False)


# --- 4. Margins Table (Question 6) ---
# We already calculated margins_2021 and margins_2026 in Cell 6
# Merge them with constituency details for Power BI
margins_combined = pd.merge(master_df, margins_2021[['ac_number', 'margin_pct']], on='ac_number')
margins_combined.rename(columns={'margin_pct': 'margin_pct_2021'}, inplace=True)

margins_combined = pd.merge(margins_combined, margins_2026[['ac_number', 'margin_pct']], on='ac_number')
margins_combined.rename(columns={'margin_pct': 'margin_pct_2026'}, inplace=True)

margins_combined.to_csv('../data/powerbi_margins.csv', index=False)

--- Top 20 Turnout Increases by Region ---
region
Chennai Metro    16
Kongu             4
Name: count, dtype: int64
